In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
# Checking preprocess.py
import sys
sys.path.append("../src")
from data.preprocess import load_ratings

df = load_ratings("../data/raw/ml-1m/ratings.dat")
df.head()

In [ ]:
df.shape

In [ ]:
# Filtering positives as per threshold (i.e. >=4)
from data.preprocess import filter_positive

positives = filter_positive(df)
positives.shape

In [ ]:
# Test train split
from data.preprocess import temporal_split
train, test = temporal_split(positives)
train.shape, test.shape

In [ ]:
# Number of users who have not rated any movie positively as per set threshold
all_users = set(df["user_id"].unique())
remaining_users = set(positives["user_id"].unique())
missing_users = all_users - remaining_users
print(missing_users)

In [ ]:
# Cross verification
df[df["user_id"] == 3598]["rating"].value_counts()

In [ ]:
df[df["user_id"] == 4486]["rating"].value_counts()

In [ ]:
# Number of movies with no positive ratings as per set threshold
all_movies = set(df["movie_id"].unique())
remaining_movies = set(positives["movie_id"].unique())
missing_movies = all_movies - remaining_movies
len(missing_movies)

In [ ]:
# Converting user and movie ids to continuos numbers
from data.preprocess import build_id_mappings
user_to_idx, movie_to_idx = build_id_mappings(positives)
len(user_to_idx), len(movie_to_idx)

In [ ]:
# Apply mappings
from data.preprocess import apply_mappings
train_mapped = apply_mappings(train, user_to_idx, movie_to_idx)
test_mapped = apply_mappings(test, user_to_idx, movie_to_idx)
train_mapped.head()

In [ ]:
train_mapped["user_id"].min(), train_mapped["user_id"].max()

In [ ]:
train_mapped["movie_id"].min(), train_mapped["movie_id"].max()

In [ ]:
assert train_mapped["user_id"].min() == 0
assert train_mapped["movie_id"].min() == 0

assert train_mapped["user_id"].max() < len(user_to_idx)
assert train_mapped["movie_id"].max() < len(movie_to_idx)

assert test_mapped["user_id"].max() < len(user_to_idx)
assert test_mapped["movie_id"].max() < len(movie_to_idx)

print("✓ Mapping sanity checks passed")

In [ ]:
# Save the processed data
from data.preprocess import save_processed
save_processed(train_mapped, test_mapped, user_to_idx, movie_to_idx, threshold=4)

In [ ]:
check = pd.read_csv("../data/processed/train.csv")
check.shape
check.head()

In [ ]:
import os
os.listdir("../data/processed")

In [ ]:
import json
with open("../data/processed/metadata.json") as f:
    print(json.load(f))

In [ ]:
positive_counts = positives.groupby("user_id").size()

print(
    "Users with exactly one positive interaction:",
    (positive_counts == 1).sum()
)

In [ ]:
movie_counts = positives.groupby("movie_id").size()

print(
    "Movies with exactly one positive interaction:",
    (movie_counts == 1).sum()
)